In [ ]:
import pandas as pd

# Load existing dataset
df = pd.read_csv('../data/dataset.csv', low_memory=False)

# Convert issue_d to datetime
df['issue_d'] = pd.to_datetime(df['issue_d'])

# Drop additional leakage columns (spec + additional found)
leakage_cols = [
    'last_pymnt_d', 'last_credit_pull_d', 'debt_settlement_flag'
]
df = df.drop(columns=leakage_cols, errors='ignore')

# Drop constant columns
constant_cols = ['pymnt_plan', 'out_prncp', 'out_prncp_inv', 'policy_code', 'hardship_flag']
df = df.drop(columns=constant_cols, errors='ignore')

# Drop unstructured / high cardinality / redundant features
unstructured_cols = ['emp_title', 'url', 'zip_code', 'title', 'sub_grade']
df = df.drop(columns=unstructured_cols, errors='ignore')

# Feature engineering: months_since_earliest_cr_line
if 'earliest_cr_line' in df.columns:
    df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')
    df['months_since_earliest_cr_line'] = ((df['issue_d'] - df['earliest_cr_line']).dt.days / 30.44).round()
    df = df.drop(columns=['earliest_cr_line'])

# Drop high missing (>40%) columns
high_missing_cols = [
    'member_id', 'desc', 'next_pymnt_d', 'mths_since_last_delinq', 'mths_since_last_record', 
    'mths_since_last_major_derog', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 
    'mths_since_recent_bc_dlq', 'mths_since_recent_revol_delinq', 'revol_bal_joint', 'sec_app_fico_range_low', 
    'sec_app_fico_range_high', 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 
    'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 
    'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med', 
    'sec_app_mths_since_last_major_derog', 'hardship_type', 'hardship_reason', 'hardship_status', 
    'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date', 'payment_plan_start_date', 
    'hardship_length', 'hardship_dpd', 'hardship_loan_status', 'orig_projected_additional_accrued_interest', 
    'hardship_payoff_balance_amount', 'hardship_last_payment_amount', 'debt_settlement_flag_date', 
    'settlement_status', 'settlement_date', 'settlement_amount', 'settlement_percentage', 'settlement_term'
]
df = df.drop(columns=high_missing_cols, errors='ignore')

# Save back to dataset.csv
df.to_csv('../data/dataset.csv', index=False)
print(f"Final dataset shape: {df.shape}")
